# Research 3 - frozen handoff and batch reliability diagnostic

Run the five cells in order: setup, mount Drive, restore, diagnostic, export.
This stage loads frozen H2, reference, and coverage adapters sequentially; it makes
no weight updates. All model actions are simulated memory operations. The 12
handoff fixtures and exact batch anchors are development diagnostics, including
supplied histories. They do not count as new generalization evidence.

One A100 80 GB; one resident model worker. Cumulative authorization: 200 Colab
units, 79.30 previously accounted. Refresh the observed balance/rate before
running. This stage has a 10-unit cap, including a 2-unit export/release reserve.
Main work is capped at 4,200 seconds and reduced by elapsed setup time.
The export cell verifies the private Drive copy and then releases the runtime.


In [ ]:
import gzip,hashlib,json,subprocess,sys,time,uuid
from pathlib import Path,PurePosixPath
from urllib.request import urlopen
SESSION_STARTED=time.monotonic()
BALANCE_AT_LAUNCH=122.83
RATE_AT_LAUNCH=6.77
PRIOR_SPEND=79.30
AUTHORIZED_TOTAL_UNITS=200
STAGE_CAP_UNITS=10
RESERVE_UNITS=2
MAX_MAIN_SECONDS=4200
assert STAGE_CAP_UNITS<=min(BALANCE_AT_LAUNCH,AUTHORIZED_TOTAL_UNITS-PRIOR_SPEND)
gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True,text=True,check=True).stdout
assert 'A100' in gpu and int(gpu.split(',')[1].split()[0])>=75000,gpu
ROOT=Path('/content/sp_lense_work')/('qwen38_handoff_reliability_'+
    time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())+'_'+uuid.uuid4().hex[:8])
ROOT.mkdir(parents=True,exist_ok=False)
SOURCE_REVISION='64833ade68fcb831a5d2da1418dc60af321cab08'
SOURCE_URL='https://raw.githubusercontent.com/Farhad-Davaripour/sp_lense/64833ade68fcb831a5d2da1418dc60af321cab08/study/03_open_agent_self_preservation/qwen38_colab/handoff_reliability_v1/source_bundle.json.gz'
SOURCE_BUNDLE_SHA256='caf7c595ad74c91cd8c37970b28b107ac46952dc1a103914f1bedc6bea710798'
with urlopen(SOURCE_URL,timeout=120) as response: packed=response.read()
assert len(packed)==42859 and hashlib.sha256(packed).hexdigest()==SOURCE_BUNDLE_SHA256
payload=json.loads(gzip.decompress(packed))
assert payload['schema_version']==1
FILES=payload['files']
for name,content in FILES.items():
    relative=PurePosixPath(name)
    assert not relative.is_absolute() and '..' not in relative.parts
    path=ROOT/'source'/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_bytes(content.encode('utf-8'))
for name,expected in json.loads(FILES['SOURCE_FREEZE.json'])['sha256'].items():
    assert hashlib.sha256((ROOT/'source'/name).read_bytes()).hexdigest()==expected,name
(ROOT/'source_bundle.json.gz').write_bytes(packed)
(ROOT/'SOURCE_BUNDLE_DOWNLOAD.json').write_text(json.dumps({
    'revision':SOURCE_REVISION,'url':SOURCE_URL,'sha256':SOURCE_BUNDLE_SHA256,
    'bytes':len(packed)},indent=2))
subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-input','-r',
                str(ROOT/'source/requirements.txt')],check=True,timeout=900)
sys.path.insert(0,str(ROOT/'source'))
print('FROZEN_DIAGNOSTIC_SOURCE_READY',ROOT,gpu,
      'authorized remaining',AUTHORIZED_TOTAL_UNITS-PRIOR_SPEND,flush=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('DRIVE_MOUNT_CONNECTED',flush=True)


In [ ]:
from restore_inputs import restore_adapters,download_base
MODEL_INPUTS=restore_adapters(ROOT)
drive.flush_and_unmount()
print('DRIVE_UNMOUNTED_BEFORE_MODEL_WORK',flush=True)
BASE_MANIFEST=download_base(ROOT)
print('H2_REFERENCE_COVERAGE_RESTORED',flush=True)


In [ ]:
exec(compile((ROOT/'source/launch.py').read_text(),
             'trusted_handoff_diagnostic.py','exec'))


In [ ]:
# Worker exit precedes derived analysis and private export.
import re,base64
for name in ('worker.py','diagnostic_worker.py'):
    for worker in ROOT.rglob(name):
        assert not subprocess.run(['pgrep','-f',re.escape(str(worker))],capture_output=True,text=True).stdout.strip()
from urllib.request import urlopen
ANALYSIS_URL='https://raw.githubusercontent.com/Farhad-Davaripour/sp_lense/79c92a40a3ae6b43f338466acb7ebe681ab7d658/study/03_open_agent_self_preservation/qwen38_colab/handoff_reliability_v1/analysis/summarize.py'
ANALYSIS_SHA='ee74b722765d5103cd8f116d3aa67eb9449e2a5e805a17b0dd0d2429a8cb2814'
with urlopen(ANALYSIS_URL,timeout=120) as response: analysis_source=response.read()
assert hashlib.sha256(analysis_source).hexdigest()==ANALYSIS_SHA
analysis_scope={'__name__':'research3_post_run_analysis'}
exec(compile(analysis_source.decode('utf-8'),'trusted_derived_analysis.py','exec'),analysis_scope)
ANALYSIS_PACKET=analysis_scope['collect'](ROOT)
analysis_scope['write_packet'](ANALYSIS_PACKET,ROOT/'review_documentation')
(ROOT/'review_documentation/ANALYSIS_SOURCE_PIN.json').write_text(json.dumps({'url':ANALYSIS_URL,'sha256':ANALYSIS_SHA},indent=2))
print('POST_RUN_ANALYSIS_READY',json.dumps({'completed':ANALYSIS_PACKET['summary']['worker_completed'],'trajectories':ANALYSIS_PACKET['summary']['evidence_files']['completed_trajectory_files'],'generation_calls':ANALYSIS_PACKET['summary']['observations']['actual_generation_call_count']}),flush=True)
print('ANALYSIS_PACKET_BEGIN'+base64.b64encode(gzip.compress(json.dumps(ANALYSIS_PACKET).encode())).decode()+'ANALYSIS_PACKET_END',flush=True)
FAST_ROOT=ROOT
EXPORT_ONLY_ROOTS=[ROOT]
RELEASE_RUNTIME_AFTER_EXPORT=True
exec(compile((ROOT/'source/export_private_runs.py').read_text(),'trusted_private_export.py','exec'))
